In [ ]:
import math
import re


def chunk_by_section(text):
    """Split Markdown at level-one headings, keeping subsections together."""
    sections = re.split(r"(?m)(?=^#\s+)", text)
    return [section.strip() for section in sections if section.strip()]


class BM25Index:
    """Small dependency-free BM25 index for documents with a content field."""

    def __init__(self, k1=1.5, b=0.75):
        self.documents = []
        self.k1 = k1
        self.b = b

    def add_document(self, document):
        if not isinstance(document.get("content"), str):
            raise ValueError("Each document must have text in its 'content' field.")
        self.documents.append(document)

    @staticmethod
    def _tokenize(text):
        return re.findall(r"\b[\w-]+\b", text.casefold())

    def search(self, query, limit=3):
        if limit <= 0 or not self.documents:
            return []

        query_terms = set(self._tokenize(query))
        tokenized_docs = [self._tokenize(doc["content"]) for doc in self.documents]
        total_docs = len(tokenized_docs)
        avg_doc_len = sum(map(len, tokenized_docs)) / total_docs

        doc_frequencies = {}
        for tokens in tokenized_docs:
            for term in set(tokens):
                doc_frequencies[term] = doc_frequencies.get(term, 0) + 1

        ranked = []
        for document, tokens in zip(self.documents, tokenized_docs):
            score = 0.0
            doc_len = len(tokens)
            for term in query_terms:
                term_frequency = tokens.count(term)
                if term_frequency == 0:
                    continue
                doc_frequency = doc_frequencies[term]
                idf = math.log(1 + (total_docs - doc_frequency + 0.5) / (doc_frequency + 0.5))
                length_norm = 1 - self.b + self.b * doc_len / avg_doc_len
                score += idf * term_frequency * (self.k1 + 1) / (
                    term_frequency + self.k1 * length_norm
                )
            if score > 0:
                ranked.append((document, score))

        return sorted(ranked, key=lambda item: item[1], reverse=True)[:limit]


# Sample input so this notebook runs as-is. Replace it with your report text.
text = """# Incident INC-2023-Q4-011

## Summary
A sample service interruption affected report processing during Q4. The team
restored processing and monitored the service after recovery.

## Follow-up
The team reviewed the processing queue and documented steps to reduce delays
if a similar interruption happens again.

# Other report

## Summary
A separate sample describes routine maintenance and a scheduled update.
"""

# 1. Split the report into sections. Each section becomes a searchable document.
chunks = chunk_by_section(text)

# 2. Add each section to the BM25 index.
store = BM25Index()
for chunk in chunks:
    store.add_document({"content": chunk})

# 3. Search. BM25 scores are relevance scores: a higher score is a better match.
results = store.search("What happened with INC-2023-Q4-011?", 3)

# 4. Print the top matching sections.
for doc, score in results:
    print(f"BM25 score: {score:.3f}\n{doc['content'][:200]}\n----")
